# Experiment 1: Pipelined KV-Cache Attention

Compare the cost (cycles and energy) of KV-cache attention with and without pipelining.

Three workloads are evaluated against the TPU v4i architecture:
- **Baseline** (`gpt3_175B_kv_cache.yaml`): full context processed in one pass
- **2-chunk pipeline** (`gpt3_175B_kv_cache_pipeline2.yaml`): context split into 2 chunks, accumulated via vector unit
- **8-chunk pipeline** (`gpt3_175B_kv_cache_pipeline8.yaml`): context split into 8 chunks, accumulated via vector unit

Each workload is mapped automatically with AccelForge's mapper.

In [115]:
from accelforge import Spec, examples
from pathlib import Path

In [116]:
def get_cycles(result):
    return float(result.latency())

def get_energy(result):
    return float(result.energy())

def get_component_energy(result, component):
    energy = result.energy(per_component=True)
    return float(energy.get(component, 0))

def get_component_cyles(result, component):
    latency = result.energy(per_component=True)
    return float(latency.get(component, 0))

## P=2 set up

In [117]:
def get_results(qk_results, sm_results, av_results, acc_results, modification, percentage):
    print(modification, percentage)
    print('cycles:')
    print('QK:', get_cycles(qk_results))
    print('SM:', get_cycles(sm_results))
    print('AV:', get_cycles(av_results))
    print('ACC:', get_cycles(acc_results))
    print()
    print('energy')
    print('QK:', get_energy(qk_results))
    print('SM:', get_energy(sm_results))
    print('AV:', get_energy(av_results))
    print('ACC:', get_energy(acc_results))
    
    # Total Pipeline Cycles
    t0_cycles = get_cycles(qk_results)
    t1_cycles = max(get_cycles(qk_results), get_cycles(sm_results))
    t2_cycles = max(get_cycles(av_results), get_cycles(sm_results))
    t3_cycles = get_cycles(av_results)
    t4_cycles = get_cycles(acc_results)
    
    print("Total Pipeline Cycles: ", t0_cycles+t1_cycles+t2_cycles+t3_cycles+t4_cycles)
    
    # Total Pipeline Energy
    t0_energy = get_energy(qk_results)
    t1_energy = max(get_energy(qk_results), get_energy(sm_results))
    t2_energy = max(get_energy(av_results), get_energy(sm_results))
    t3_energy = get_energy(av_results)
    t4_energy = get_energy(acc_results)
    
    print("Total Pipeline Energy: ", t0_energy+t1_energy+t2_energy+t3_energy+t4_energy)
    print()
    print()

## Baseline

In [118]:
qk_spec_base = Spec.from_yaml(
    "../arches/tpu_v4i_with_VPU_QK.yaml",
    "../workloads/C_2/flash_attention_C_2_QK.yaml"
)
qk_results_base = qk_spec_base.map_workload_to_arch()

sm_spec_base = Spec.from_yaml(
    "../arches/tpu_v4i_with_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_SM.yaml"
)
sm_results_base = sm_spec_base.map_workload_to_arch()

av_spec_base = Spec.from_yaml(
    "../arches/tpu_v4i_with_VPU_AV.yaml",
    "../workloads/C_2/flash_attention_C_2_AV.yaml"
)
av_results_base = av_spec_base.map_workload_to_arch()

acc_spec_base = Spec.from_yaml(
    "../arches/tpu_v4i_only_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_ACC.yaml"
)
acc_results_base = acc_spec_base.map_workload_to_arch()

Getting energy, latency, and leak power for components running QK_1: 100%|████████████████| 1/1 [00:00<00:00,  3.34it/s]
Generating jobs:   0%|                                                                            | 0/1 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum QK_1: 0it [00:00, ?it/s]

Generating pmapping templates for compute MXU Einsum QK_1: 0it [00:00, ?it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 3it [00:00, 28.03it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 8it [00:00, 37.99it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 12it [00:00, 38.34it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 17it [00:00, 39.22it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 21it [00:00, 39.12it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 26it [00:00, 40.53it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 32it [00:00, 40.64it/s]
Generating jobs

Einsum QK_1 has 32 pmapping jobs:
	0	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	1	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-h  T-m  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	2	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	3	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] T-m  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	4	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-m_chunk  S-reuse_

Compressing pmappings: 100%|██████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 45.53it/s]


Dirty joining with resource usage <= 1.02× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 74.20it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>energy=2.84e-06
Final clean join.


Dirty pruning pmappings: 100%|██████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1189.87it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 1 -> 1 (100.00% kept) pmappings


Final consolidate: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 5769.33it/s]


Dirty joining mapping(s) valid & optimal! Returning...


# GLB Capacity

## 25% GLB Capacity

In [119]:
qk_spec_25_cap = Spec.from_yaml(
    "../arches/capacity/25_percent/tpu_v4i_with_VPU_QK.yaml",
    "../workloads/C_2/flash_attention_C_2_QK.yaml"
)
qk_results_25_cap = qk_spec_25_cap.map_workload_to_arch()

sm_spec_25_cap = Spec.from_yaml(
    "../arches/capacity/25_percent/tpu_v4i_with_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_SM.yaml"
)
sm_results_25_cap = sm_spec_25_cap.map_workload_to_arch()

av_spec_25_cap = Spec.from_yaml(
    "../arches/capacity/25_percent/tpu_v4i_with_VPU_AV.yaml",
    "../workloads/C_2/flash_attention_C_2_AV.yaml"
)
av_results_25_cap = av_spec_25_cap.map_workload_to_arch()

acc_spec_25_cap = Spec.from_yaml(
    "../arches/capacity/25_percent/tpu_v4i_only_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_ACC.yaml"
)
acc_results_25_cap = acc_spec_25_cap.map_workload_to_arch()


Getting energy, latency, and leak power for components running QK_1: 100%|████████████████| 1/1 [00:00<00:00, 14.08it/s]
Generating jobs:   0%|                                                                            | 0/1 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum QK_1: 0it [00:00, ?it/s]

Generating pmapping templates for compute MXU Einsum QK_1: 0it [00:00, ?it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 12it [00:00, 114.68it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 32it [00:00, 104.45it/s]
Generating jobs: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.77it/s]


Einsum QK_1 has 32 pmapping jobs:
	0	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	1	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-h  T-m  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	2	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	3	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] T-m  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	4	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-m_chunk  S-reuse_

Compressing pmappings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 468.79it/s]


Dirty joining with resource usage <= 1.02× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 839.20it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>energy=2.84e-06
Final clean join.


Dirty pruning pmappings: 100%|██████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1206.65it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 1 -> 1 (100.00% kept) pmappings


Final consolidate: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 7839.82it/s]


Dirty joining mapping(s) valid & optimal! Returning...


## 50% GLB Capacity

In [120]:
qk_spec_50_cap = Spec.from_yaml(
    "../arches/capacity/50_percent/tpu_v4i_with_VPU_QK.yaml",
    "../workloads/C_2/flash_attention_C_2_QK.yaml"
)
qk_results_50_cap = qk_spec_50_cap.map_workload_to_arch()

sm_spec_50_cap = Spec.from_yaml(
    "../arches/capacity/50_percent/tpu_v4i_with_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_SM.yaml"
)
sm_results_50_cap = sm_spec_50_cap.map_workload_to_arch()

av_spec_50_cap = Spec.from_yaml(
    "../arches/capacity/50_percent/tpu_v4i_with_VPU_AV.yaml",
    "../workloads/C_2/flash_attention_C_2_AV.yaml"
)
av_results_50_cap = av_spec_50_cap.map_workload_to_arch()

acc_spec_50_cap = Spec.from_yaml(
    "../arches/capacity/50_percent/tpu_v4i_only_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_ACC.yaml"
)
acc_results_50_cap = acc_spec_50_cap.map_workload_to_arch()


Getting energy, latency, and leak power for components running QK_1: 100%|████████████████| 1/1 [00:00<00:00, 25.43it/s]
Generating jobs:   0%|                                                                            | 0/1 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum QK_1: 0it [00:00, ?it/s]

Generating pmapping templates for compute MXU Einsum QK_1: 0it [00:00, ?it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 14it [00:00, 135.47it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 32it [00:00, 56.05it/s] 
Generating jobs: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.61it/s]


Einsum QK_1 has 32 pmapping jobs:
	0	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	1	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-h  T-m  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	2	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	3	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] T-m  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	4	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-m_chunk  S-reuse_

Compressing pmappings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 207.46it/s]


Dirty joining with resource usage <= 1.02× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 828.10it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>energy=2.84e-06
Final clean join.


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 783.25it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 1 -> 1 (100.00% kept) pmappings


Final consolidate: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 4452.55it/s]


Dirty joining mapping(s) valid & optimal! Returning...


## 200% GLB Capacity

In [121]:
qk_spec_200_cap = Spec.from_yaml(
    "../arches/capacity/200_percent/tpu_v4i_with_VPU_QK.yaml",
    "../workloads/C_2/flash_attention_C_2_QK.yaml"
)
qk_results_200_cap = qk_spec_200_cap.map_workload_to_arch()

sm_spec_200_cap = Spec.from_yaml(
    "../arches/capacity/200_percent/tpu_v4i_with_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_SM.yaml"
)
sm_results_200_cap = sm_spec_200_cap.map_workload_to_arch()

av_spec_200_cap = Spec.from_yaml(
    "../arches/capacity/200_percent/tpu_v4i_with_VPU_AV.yaml",
    "../workloads/C_2/flash_attention_C_2_AV.yaml"
)
av_results_200_cap = av_spec_200_cap.map_workload_to_arch()

acc_spec_200_cap = Spec.from_yaml(
    "../arches/capacity/200_percent/tpu_v4i_only_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_ACC.yaml"
)
acc_results_200_cap = acc_spec_200_cap.map_workload_to_arch()


Getting energy, latency, and leak power for components running QK_1: 100%|████████████████| 1/1 [00:00<00:00, 24.46it/s]
Generating jobs:   0%|                                                                            | 0/1 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum QK_1: 0it [00:00, ?it/s]

Generating pmapping templates for compute MXU Einsum QK_1: 0it [00:00, ?it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 12it [00:00, 114.73it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 32it [00:00, 107.41it/s]
Generating jobs: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.86it/s]


Einsum QK_1 has 32 pmapping jobs:
	0	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	1	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-h  T-m  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	2	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	3	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] T-m  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	4	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-m_chunk  S-reuse_

Compressing pmappings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 414.33it/s]


Dirty joining with resource usage <= 1.02× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 970.01it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>energy=2.84e-06
Final clean join.


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 719.43it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 1 -> 1 (100.00% kept) pmappings


Final consolidate: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 6204.59it/s]


Dirty joining mapping(s) valid & optimal! Returning...


## Comparisons

In [123]:
print('\nbase results:')
get_results(qk_results_base, sm_results_base, av_results_base, acc_results_base, 'base', '100%')

print('\n25 results:')
get_results(qk_results_25_cap, sm_results_25_cap, av_results_25_cap, acc_results_25_cap, 'capacity', '25%')

print('\n50 results:')
get_results(qk_results_50_cap, sm_results_50_cap, av_results_50_cap, acc_results_50_cap, 'capacity', '50%')

print('\n200 results:')
get_results(qk_results_200_cap, sm_results_200_cap, av_results_200_cap, acc_results_200_cap, 'capacity', '200%')

"""
The reason this happens is because all our tensors fit within
the GLB capacity
"""


base results:
base 100%
cycles:
QK: 0.00011017840006388724
SM: 1.5603809515596367e-05
AV: 0.00011017840006388724
ACC: 1.2190476184059662e-07

energy
QK: 0.00440040966177228
SM: 0.00013484449258408302
AV: 0.004392215046662048
ACC: 2.8405923653402333e-06
Total Pipeline Cycles:  0.00044083550501738955
Total Pipeline Energy:  0.017588090009233996



25 results:
capacity 25%
cycles:
QK: 0.00011017840006388724
SM: 1.5603809515596367e-05
AV: 0.00011017840006388724
ACC: 1.2190476184059662e-07

energy
QK: 0.00440040966177228
SM: 0.00013484449258408302
AV: 0.004392215046662048
ACC: 2.8405923653402333e-06
Total Pipeline Cycles:  0.00044083550501738955
Total Pipeline Energy:  0.017588090009233996



50 results:
capacity 50%
cycles:
QK: 0.00011017840006388724
SM: 1.5603809515596367e-05
AV: 0.00011017840006388724
ACC: 1.2190476184059662e-07

energy
QK: 0.00440040966177228
SM: 0.00013484449258408302
AV: 0.004392215046662048
ACC: 2.8405923653402333e-06
Total Pipeline Cycles:  0.00044083550501738955
T

'\nThe reason this happens is because all our tensors fit within\nthe GLB capacity\n'

In [124]:
# Compare max tensor footprint vs GLB sizes
M_CHUNK, H, E = 4096, 128, 128
bytes_per_val = 1  # 8-bit

# Largest tensor in QK stage: K_1[M_CHUNK, H, E]
k1_bytes = M_CHUNK * H * E * bytes_per_val
print(f"K_1 size: {k1_bytes / 1e6:.1f} MB")

glbs = {
    "25%":  1024*1024*128*2,
    "50%":  1024*1024*128*4,
    "100%": 1024*1024*128*8,
    "200%": 1024*1024*128*16,
}
for name, size in glbs.items():
    print(f"GLB {name}: {size/1e6:.0f} MB  — K_1 fits: {k1_bytes < size}")

K_1 size: 67.1 MB
GLB 25%: 268 MB  — K_1 fits: True
GLB 50%: 537 MB  — K_1 fits: True
GLB 100%: 1074 MB  — K_1 fits: True
GLB 200%: 2147 MB  — K_1 fits: True


# Bandwidth

## 25%

In [125]:
qk_spec_25_band = Spec.from_yaml(
    "../arches/bandwidth/25_percent/tpu_v4i_with_VPU_QK.yaml",
    "../workloads/C_2/flash_attention_C_2_QK.yaml"
)
qk_results_25_band = qk_spec_25_band.map_workload_to_arch()

sm_spec_25_band = Spec.from_yaml(
    "../arches/bandwidth/25_percent/tpu_v4i_with_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_SM.yaml"
)
sm_results_25_band = sm_spec_25_band.map_workload_to_arch()

av_spec_25_band = Spec.from_yaml(
    "../arches/bandwidth/25_percent/tpu_v4i_with_VPU_AV.yaml",
    "../workloads/C_2/flash_attention_C_2_AV.yaml"
)
av_results_25_band = av_spec_25_band.map_workload_to_arch()

acc_spec_25_band = Spec.from_yaml(
    "../arches/bandwidth/25_percent/tpu_v4i_only_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_ACC.yaml"
)
acc_results_25_band = acc_spec_25_band.map_workload_to_arch()


Getting energy, latency, and leak power for components running QK_1: 100%|████████████████| 1/1 [00:00<00:00, 20.17it/s]
Generating jobs:   0%|                                                                            | 0/1 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum QK_1: 0it [00:00, ?it/s]

Generating pmapping templates for compute MXU Einsum QK_1: 0it [00:00, ?it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 12it [00:00, 119.95it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 32it [00:00, 116.60it/s]
Generating jobs: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.02it/s]


Einsum QK_1 has 32 pmapping jobs:
	0	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	1	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-h  T-m  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	2	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	3	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] T-m  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	4	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-m_chunk  S-reuse_

Compressing pmappings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 354.52it/s]


Dirty joining with resource usage <= 1.02× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|██████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1146.30it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>energy=2.84e-06
Final clean join.


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 927.94it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 1 -> 1 (100.00% kept) pmappings


Final consolidate: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 5377.31it/s]


Dirty joining mapping(s) valid & optimal! Returning...


## 50%

In [126]:
qk_spec_50_band = Spec.from_yaml(
    "../arches/bandwidth/50_percent/tpu_v4i_with_VPU_QK.yaml",
    "../workloads/C_2/flash_attention_C_2_QK.yaml"
)
qk_results_50_band = qk_spec_50_band.map_workload_to_arch()

sm_spec_50_band = Spec.from_yaml(
    "../arches/bandwidth/50_percent/tpu_v4i_with_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_SM.yaml"
)
sm_results_50_band = sm_spec_50_band.map_workload_to_arch()

av_spec_50_band = Spec.from_yaml(
    "../arches/bandwidth/50_percent/tpu_v4i_with_VPU_AV.yaml",
    "../workloads/C_2/flash_attention_C_2_AV.yaml"
)
av_results_50_band = av_spec_50_band.map_workload_to_arch()

acc_spec_50_band = Spec.from_yaml(
    "../arches/bandwidth/50_percent/tpu_v4i_only_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_ACC.yaml"
)
acc_results_50_band = acc_spec_50_band.map_workload_to_arch()


Getting energy, latency, and leak power for components running QK_1: 100%|████████████████| 1/1 [00:00<00:00, 22.58it/s]
Generating jobs:   0%|                                                                            | 0/1 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum QK_1: 0it [00:00, ?it/s]

Generating pmapping templates for compute MXU Einsum QK_1: 0it [00:00, ?it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 13it [00:00, 124.06it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 32it [00:00, 125.92it/s]
Generating jobs: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.18it/s]


Einsum QK_1 has 32 pmapping jobs:
	0	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	1	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-h  T-m  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	2	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	3	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] T-m  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	4	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-m_chunk  S-reuse_

Compressing pmappings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 362.67it/s]


Dirty joining with resource usage <= 1.02× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 998.17it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>energy=2.84e-06
Final clean join.


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 922.43it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 1 -> 1 (100.00% kept) pmappings


Final consolidate: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 6864.65it/s]


Dirty joining mapping(s) valid & optimal! Returning...


## 200%

In [127]:
qk_spec_200_band = Spec.from_yaml(
    "../arches/bandwidth/200_percent/tpu_v4i_with_VPU_QK.yaml",
    "../workloads/C_2/flash_attention_C_2_QK.yaml"
)
qk_results_200_band = qk_spec_200_band.map_workload_to_arch()

sm_spec_200_band = Spec.from_yaml(
    "../arches/bandwidth/200_percent/tpu_v4i_with_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_SM.yaml"
)
sm_results_200_band = sm_spec_200_band.map_workload_to_arch()

av_spec_200_band = Spec.from_yaml(
    "../arches/bandwidth/200_percent/tpu_v4i_with_VPU_AV.yaml",
    "../workloads/C_2/flash_attention_C_2_AV.yaml"
)
av_results_200_band = av_spec_200_band.map_workload_to_arch()

acc_spec_200_band = Spec.from_yaml(
    "../arches/bandwidth/200_percent/tpu_v4i_only_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_ACC.yaml"
)
acc_results_200_band = acc_spec_200_band.map_workload_to_arch()

Getting energy, latency, and leak power for components running QK_1: 100%|████████████████| 1/1 [00:00<00:00, 25.99it/s]
Generating jobs:   0%|                                                                            | 0/1 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum QK_1: 0it [00:00, ?it/s]

Generating pmapping templates for compute MXU Einsum QK_1: 0it [00:00, ?it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 12it [00:00, 113.28it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 32it [00:00, 117.24it/s]
Generating jobs: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.06it/s]


Einsum QK_1 has 32 pmapping jobs:
	0	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	1	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-h  T-m  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	2	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	3	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] T-m  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	4	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-m_chunk  S-reuse_

Compressing pmappings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 362.67it/s]


Dirty joining with resource usage <= 1.02× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|██████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1138.52it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>energy=2.84e-06
Final clean join.


Dirty pruning pmappings: 100%|██████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1100.00it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 1 -> 1 (100.00% kept) pmappings


Final consolidate: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 6043.67it/s]


Dirty joining mapping(s) valid & optimal! Returning...


## Comparison

In [128]:
print('\nbase results:')
get_results(qk_results_base, sm_results_base, av_results_base, acc_results_base, 'base', '100%')

print('\n25 results:')
get_results(qk_results_25_band, sm_results_25_band, av_results_25_band, acc_results_25_band, 'bandwidth', '25%')

print('\n50 results:')
get_results(qk_results_50_band, sm_results_50_band, av_results_50_band, acc_results_50_band, 'bandwidth', '50%')

print('\n200 results:')
get_results(qk_results_200_band, sm_results_200_band, av_results_200_band, acc_results_200_band, 'bandwidth', '200%')



base results:
base 100%
cycles:
QK: 0.00011017840006388724
SM: 1.5603809515596367e-05
AV: 0.00011017840006388724
ACC: 1.2190476184059662e-07

energy
QK: 0.00440040966177228
SM: 0.00013484449258408302
AV: 0.004392215046662048
ACC: 2.8405923653402333e-06
Total Pipeline Cycles:  0.00044083550501738955
Total Pipeline Energy:  0.017588090009233996



25 results:
bandwidth 25%
cycles:
QK: 0.00044071360025554895
SM: 1.5603809515596367e-05
AV: 0.00044071360025554895
ACC: 3.202084712938813e-07

energy
QK: 0.00440040966177228
SM: 0.00013484449258408302
AV: 0.004392215046662048
ACC: 2.8405923653402333e-06
Total Pipeline Cycles:  0.0017631746094934897
Total Pipeline Energy:  0.017588090009233996



50 results:
bandwidth 50%
cycles:
QK: 0.00022035680012777448
SM: 1.5603809515596367e-05
AV: 0.00022035680012777448
ACC: 1.6010423564694065e-07

energy
QK: 0.00440040966177228
SM: 0.00013484449258408302
AV: 0.004392215046662048
ACC: 2.8405923653402333e-06
Total Pipeline Cycles:  0.0008815873047467448
To

In [129]:
# Check whether the workload is bandwidth-bound or compute-bound
# If compute time >> memory transfer time, bandwidth doesn't matter

M_CHUNK, H, E = 4096, 128, 128
bytes_per_val = 1  # 8-bit

# Total bytes transferred for QK stage (read Q, K_1; write QK_1)
# Q[1, 1, H, E], K_1[1, M_CHUNK, H, E], QK_1[1, 1, M_CHUNK, H]
q_bytes   = 1 * 1 * H * E * bytes_per_val
k1_bytes  = 1 * M_CHUNK * H * E * bytes_per_val
qk1_bytes = 1 * 1 * M_CHUNK * H * bytes_per_val
total_bw_bytes = q_bytes + k1_bytes + qk1_bytes
print(f"QK stage total data transferred: {total_bw_bytes / 1e6:.1f} MB")

# MXU compute: Q * K_1 = 1 x H x E * M_CHUNK ops → M_CHUNK * H * E MACs
mxu_ops = M_CHUNK * H * E
mxu_throughput = 128 * 128 * 2 * 1.05e9  # 128x128 MXU @ 1.05 GHz (ops/s)
compute_time_s = mxu_ops / mxu_throughput
print(f"QK compute time: {compute_time_s * 1e6:.3f} us")

bandwidths = {
    "25%  (512 GB/s)":  512e9,
    "50%  (1024 GB/s)": 1024e9,
    "100% (2048 GB/s)": 2048e9,
    "200% (4096 GB/s)": 4096e9,
}
print()
for name, bw in bandwidths.items():
    mem_time_s = total_bw_bytes / bw
    print(f"BW {name}: mem transfer = {mem_time_s * 1e6:.3f} us  — "
          f"{'MEMORY-BOUND' if mem_time_s > compute_time_s else 'COMPUTE-BOUND (BW doesnt matter)'}")

""""
Got to figure out how to make this rely on GLB instead of DRAM
The workload is memory-bound at DRAM, not GLB. Changing GLB bandwidth
(512→4096 GB/s) has no effect because the DRAM bottleneck dominates
everything downstream. To actually see a difference, you'd need to sweep
MainMemory bandwidth instead.

The reason we are compute-bound is because we have high reuse
which allows. If we don't have flash attention, we need ot compute our entire S
If we do have flash atteniton, we can partioin our softmax computation which
gives us more opportunities for reuse. A lot of our intermediate stuff
is put in our buffer allowing us to fuse through our softmax



"""

QK stage total data transferred: 67.6 MB
QK compute time: 1.950 us

BW 25%  (512 GB/s): mem transfer = 132.128 us  — MEMORY-BOUND
BW 50%  (1024 GB/s): mem transfer = 66.064 us  — MEMORY-BOUND
BW 100% (2048 GB/s): mem transfer = 33.032 us  — MEMORY-BOUND
BW 200% (4096 GB/s): mem transfer = 16.516 us  — MEMORY-BOUND


'"\nGot to figure out how to make this rely on GLB instead of DRAM\nThe workload is memory-bound at DRAM, not GLB. Changing GLB bandwidth\n(512→4096 GB/s) has no effect because the DRAM bottleneck dominates\neverything downstream. To actually see a difference, you\'d need to sweep\nMainMemory bandwidth instead.\n\nThe reason we are compute-bound is because we have high reuse\nwhich allows. If we don\'t have flash attention, we need ot compute our entire S\nIf we do have flash atteniton, we can partioin our softmax computation which\ngives us more opportunities for reuse. A lot of our intermediate stuff\nis put in our buffer allowing us to fuse through our softmax\n\n\n\n'

# Compute Throughput

## 25%

In [130]:
qk_spec_25_mxu = Spec.from_yaml(
    "../arches/mxu/25_percent/tpu_v4i_with_VPU_QK.yaml",
    "../workloads/C_2/flash_attention_C_2_QK.yaml"
)
qk_results_25_mxu = qk_spec_25_mxu.map_workload_to_arch()

sm_spec_25_mxu = Spec.from_yaml(
    "../arches/mxu/25_percent/tpu_v4i_with_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_SM.yaml"
)
sm_results_25_mxu = sm_spec_25_mxu.map_workload_to_arch()

av_spec_25_mxu = Spec.from_yaml(
    "../arches/mxu/25_percent/tpu_v4i_with_VPU_AV.yaml",
    "../workloads/C_2/flash_attention_C_2_AV.yaml"
)
av_results_25_mxu = av_spec_25_mxu.map_workload_to_arch()

acc_spec_25_mxu = Spec.from_yaml(
    "../arches/mxu/25_percent/tpu_v4i_only_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_ACC.yaml"
)
acc_results_25_mxu = acc_spec_25_mxu.map_workload_to_arch()


Getting energy, latency, and leak power for components running QK_1: 100%|████████████████| 1/1 [00:00<00:00, 26.47it/s]
Generating jobs:   0%|                                                                            | 0/1 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum QK_1: 0it [00:00, ?it/s]

Generating pmapping templates for compute MXU Einsum QK_1: 0it [00:00, ?it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 13it [00:00, 123.47it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 32it [00:00, 119.69it/s]
Generating jobs: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.10it/s]


Einsum QK_1 has 32 pmapping jobs:
	0	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	1	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-h  T-m  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	2	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	3	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] T-m  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	4	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-m_chunk  S-reuse_

Compressing pmappings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 436.18it/s]


Dirty joining with resource usage <= 1.02× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 928.77it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>energy=2.84e-06
Final clean join.


Dirty pruning pmappings: 100%|██████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1010.92it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 1 -> 1 (100.00% kept) pmappings


Final consolidate: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 5874.38it/s]


Dirty joining mapping(s) valid & optimal! Returning...


## 50%

In [131]:
qk_spec_50_mxu = Spec.from_yaml(
    "../arches/mxu/50_percent/tpu_v4i_with_VPU_QK.yaml",
    "../workloads/C_2/flash_attention_C_2_QK.yaml"
)
qk_results_50_mxu = qk_spec_50_mxu.map_workload_to_arch()

sm_spec_50_mxu = Spec.from_yaml(
    "../arches/mxu/50_percent/tpu_v4i_with_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_SM.yaml"
)
sm_results_50_mxu = sm_spec_50_mxu.map_workload_to_arch()

av_spec_50_mxu = Spec.from_yaml(
    "../arches/mxu/50_percent/tpu_v4i_with_VPU_AV.yaml",
    "../workloads/C_2/flash_attention_C_2_AV.yaml"
)
av_results_50_mxu = av_spec_50_mxu.map_workload_to_arch()

acc_spec_50_mxu = Spec.from_yaml(
    "../arches/mxu/50_percent/tpu_v4i_only_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_ACC.yaml"
)
acc_results_50_mxu = acc_spec_50_mxu.map_workload_to_arch()


Getting energy, latency, and leak power for components running QK_1: 100%|████████████████| 1/1 [00:00<00:00, 27.37it/s]
Generating jobs:   0%|                                                                            | 0/1 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum QK_1: 0it [00:00, ?it/s]

Generating pmapping templates for compute MXU Einsum QK_1: 0it [00:00, ?it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 13it [00:00, 123.33it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 32it [00:00, 119.17it/s]
Generating jobs: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.06it/s]


Einsum QK_1 has 32 pmapping jobs:
	0	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	1	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-h  T-m  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	2	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	3	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] T-m  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	4	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-m_chunk  S-reuse_

Compressing pmappings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 407.29it/s]


Dirty joining with resource usage <= 1.02× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 849.05it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>energy=2.84e-06
Final clean join.


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 754.64it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 1 -> 1 (100.00% kept) pmappings


Final consolidate: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 4793.49it/s]


Dirty joining mapping(s) valid & optimal! Returning...


## 200%

In [132]:
qk_spec_200_mxu = Spec.from_yaml(
    "../arches/mxu/200_percent/tpu_v4i_with_VPU_QK.yaml",
    "../workloads/C_2/flash_attention_C_2_QK.yaml"
)
qk_results_200_mxu = qk_spec_200_mxu.map_workload_to_arch()

sm_spec_200_mxu = Spec.from_yaml(
    "../arches/mxu/200_percent/tpu_v4i_with_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_SM.yaml"
)
sm_results_200_mxu = sm_spec_200_mxu.map_workload_to_arch()

av_spec_200_mxu = Spec.from_yaml(
    "../arches/mxu/200_percent/tpu_v4i_with_VPU_AV.yaml",
    "../workloads/C_2/flash_attention_C_2_AV.yaml"
)
av_results_200_mxu = av_spec_200_mxu.map_workload_to_arch()

acc_spec_200_mxu = Spec.from_yaml(
    "../arches/mxu/200_percent/tpu_v4i_only_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_ACC.yaml"
)
acc_results_200_mxu = acc_spec_200_mxu.map_workload_to_arch()


Getting energy, latency, and leak power for components running QK_1: 100%|████████████████| 1/1 [00:00<00:00, 22.40it/s]
Generating jobs:   0%|                                                                            | 0/1 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum QK_1: 0it [00:00, ?it/s]

Generating pmapping templates for compute MXU Einsum QK_1: 0it [00:00, ?it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 13it [00:00, 121.12it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 32it [00:00, 118.95it/s]
Generating jobs: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  3.11it/s]


Einsum QK_1 has 32 pmapping jobs:
	0	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	1	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-h  T-m  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	2	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	3	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] T-m  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	4	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-m_chunk  S-reuse_

Compressing pmappings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 401.06it/s]


Dirty joining with resource usage <= 1.02× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|███████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 681.67it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>energy=2.84e-06
Final clean join.


Dirty pruning pmappings: 100%|██████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1110.19it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 1 -> 1 (100.00% kept) pmappings


Final consolidate: 100%|████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 7108.99it/s]


Dirty joining mapping(s) valid & optimal! Returning...


## Comparison

In [133]:
print('\nbase results:')
get_results(qk_results_base, sm_results_base, av_results_base, acc_results_base, 'base', '100%')

print('\n25 results:')
get_results(qk_results_25_mxu, sm_results_25_mxu, av_results_25_mxu, acc_results_25_mxu, 'compute tp', '25%')

print('\n50 results:')
get_results(qk_results_50_mxu, sm_results_50_mxu, av_results_50_mxu, acc_results_50_mxu, 'compute tp', '50%')

print('\n200 results:')
get_results(qk_results_200_mxu, sm_results_200_mxu, av_results_200_mxu, acc_results_200_mxu, 'compute tp', '200%')


base results:
base 100%
cycles:
QK: 0.00011017840006388724
SM: 1.5603809515596367e-05
AV: 0.00011017840006388724
ACC: 1.2190476184059662e-07

energy
QK: 0.00440040966177228
SM: 0.00013484449258408302
AV: 0.004392215046662048
ACC: 2.8405923653402333e-06
Total Pipeline Cycles:  0.00044083550501738955
Total Pipeline Energy:  0.017588090009233996



25 results:
compute tp 25%
cycles:
QK: 0.00011017840006388724
SM: 6.241523806238547e-05
AV: 0.00011017840006388724
ACC: 4.876190473623865e-07

energy
QK: 0.00440040966177228
SM: 0.00013484449258408302
AV: 0.004392215046662048
ACC: 2.836725741340233e-06
Total Pipeline Cycles:  0.00044120121930291134
Total Pipeline Energy:  0.017588086142609996



50 results:
compute tp 50%
cycles:
QK: 0.00011017840006388724
SM: 3.1207619031192735e-05
AV: 0.00011017840006388724
ACC: 2.4380952368119324e-07

energy
QK: 0.00440040966177228
SM: 0.00013484449258408302
AV: 0.004392215046662048
ACC: 2.836725741340233e-06
Total Pipeline Cycles:  0.00044095740977923015
T

# bandwidth/mxu cycle checks

In [134]:
bandwidths = [qk_results_200_band, sm_results_200_band, av_results_200_band, acc_results_200_band]
print('bandwidth; 200%')
for b in bandwidths:
    print(get_cycles(b))
    print()

print()

print('mxus; 200%')
mxus = [qk_results_200_mxu, sm_results_200_mxu, av_results_200_mxu, acc_results_200_mxu]
for m in mxus:
    print(get_cycles(m))
    print()


bandwidth; 200%
5.508920003194362e-05

1.5603809515596367e-05

5.508920003194362e-05

1.2190476184059662e-07


mxus; 200%
0.00011017840006388724

7.801904757798184e-06

0.00011017840006388724

8.005211782347033e-08



In [135]:
def print_traffic_breakdown(result, label=""):
    """Print per-component energy and latency breakdown for a mapping result."""
    if label:
        print(f"=== {label} ===")

    # Per-component energy
    energy_breakdown = result.energy(per_component=True)
    total_energy = float(result.energy())
    print("Energy breakdown:")
    for comp, val in sorted(energy_breakdown.items(), key=lambda x: -float(x[1])):
        fval = float(val)
        pct = 100 * fval / total_energy if total_energy > 0 else 0
        print(f"  {comp:<20s}: {fval:.4e} J  ({pct:5.1f}%)")

    # Per-component latency
    latency_breakdown = result.latency(per_component=True)
    total_latency = float(result.latency())
    print("Latency breakdown:")
    for comp, val in sorted(latency_breakdown.items(), key=lambda x: -float(x[1])):
        fval = float(val)
        pct = 100 * fval / total_latency if total_latency > 0 else 0
        print(f"  {comp:<20s}: {fval:.4e} s  ({pct:5.1f}%)")
    print()


def print_pipeline_breakdown(qk_results, sm_results, av_results, acc_results, label=""):
    """Print traffic breakdown for all 4 pipeline stages."""
    if label:
        print(f"{'='*10} {label} {'='*10}")
    print_traffic_breakdown(qk_results, "QK stage")
    print_traffic_breakdown(sm_results, "SM (softmax) stage")
    print_traffic_breakdown(av_results, "AV stage")
    print_traffic_breakdown(acc_results, "ACC (accumulate) stage")


# Run on baseline to see where time/energy actually goes
print_pipeline_breakdown(qk_results_base, sm_results_base, av_results_base, acc_results_base, "Baseline")


========== Baseline ==========
=== QK stage ===
Energy breakdown:
  MainMemory          : 3.8046e-03 J  ( 86.5%)
  MxuBuffer           : 5.8197e-04 J  ( 13.2%)
  GlobalBuffer        : 8.1946e-06 J  (  0.2%)
  MXU                 : 5.6371e-06 J  (  0.1%)
  ScalarBuffer        : 0.0000e+00 J  (  0.0%)
  ScalarUnit          : 0.0000e+00 J  (  0.0%)
Latency breakdown:
  MainMemory          : 1.1018e-04 s  (100.0%)
  MXU                 : 3.9010e-06 s  (  3.5%)
  GlobalBuffer        : 2.5600e-07 s  (  0.2%)
  MxuBuffer           : 0.0000e+00 s  (  0.0%)

=== SM (softmax) stage ===
Energy breakdown:
  MainMemory          : 8.8472e-05 J  ( 65.6%)
  GlobalBuffer        : 2.5669e-05 J  ( 19.0%)
  ScalarBuffer        : 1.8187e-05 J  ( 13.5%)
  ScalarUnit          : 2.5166e-06 J  (  1.9%)
  MxuBuffer           : 0.0000e+00 J  (  0.0%)
  MXU                 : 0.0000e+00 J  (  0.0%)
  VpuBuffer           : 0.0000e+00 J  (  0.0%)
  VPU                 : 0.0000e+00 J  (  0.0%)
Latency breakdown:
  Sc

In [136]:
print_pipeline_breakdown(qk_results_200_mxu, sm_results_200_mxu, av_results_200_mxu, acc_results_200_mxu, "200 mxu")


========== 200 mxu ==========
=== QK stage ===
Energy breakdown:
  MainMemory          : 3.8046e-03 J  ( 86.5%)
  MxuBuffer           : 5.8197e-04 J  ( 13.2%)
  GlobalBuffer        : 8.1946e-06 J  (  0.2%)
  MXU                 : 5.6371e-06 J  (  0.1%)
  ScalarBuffer        : 0.0000e+00 J  (  0.0%)
  ScalarUnit          : 0.0000e+00 J  (  0.0%)
Latency breakdown:
  MainMemory          : 1.1018e-04 s  (100.0%)
  MXU                 : 1.9505e-06 s  (  1.8%)
  GlobalBuffer        : 2.5600e-07 s  (  0.2%)
  MxuBuffer           : 0.0000e+00 s  (  0.0%)

=== SM (softmax) stage ===
Energy breakdown:
  MainMemory          : 8.8472e-05 J  ( 65.6%)
  GlobalBuffer        : 2.5669e-05 J  ( 19.0%)
  ScalarBuffer        : 1.8187e-05 J  ( 13.5%)
  ScalarUnit          : 2.5166e-06 J  (  1.9%)
  MxuBuffer           : 0.0000e+00 J  (  0.0%)
  MXU                 : 0.0000e+00 J  (  0.0%)
  VpuBuffer           : 0.0000e+00 J  (  0.0%)
  VPU                 : 0.0000e+00 J  (  0.0%)
Latency breakdown:
  Sca

In [137]:
print_pipeline_breakdown(qk_results_200_band, sm_results_200_band, av_results_200_band, acc_results_200_band, "200 band")


========== 200 band ==========
=== QK stage ===
Energy breakdown:
  MainMemory          : 3.8046e-03 J  ( 86.5%)
  MxuBuffer           : 5.8197e-04 J  ( 13.2%)
  GlobalBuffer        : 8.1946e-06 J  (  0.2%)
  MXU                 : 5.6371e-06 J  (  0.1%)
  ScalarBuffer        : 0.0000e+00 J  (  0.0%)
  ScalarUnit          : 0.0000e+00 J  (  0.0%)
Latency breakdown:
  MainMemory          : 5.5089e-05 s  (100.0%)
  MXU                 : 3.9010e-06 s  (  7.1%)
  GlobalBuffer        : 1.2800e-07 s  (  0.2%)
  MxuBuffer           : 0.0000e+00 s  (  0.0%)

=== SM (softmax) stage ===
Energy breakdown:
  MainMemory          : 8.8472e-05 J  ( 65.6%)
  GlobalBuffer        : 2.5669e-05 J  ( 19.0%)
  ScalarBuffer        : 1.8187e-05 J  ( 13.5%)
  ScalarUnit          : 2.5166e-06 J  (  1.9%)
  MxuBuffer           : 0.0000e+00 J  (  0.0%)
  MXU                 : 0.0000e+00 J  (  0.0%)
  VpuBuffer           : 0.0000e+00 J  (  0.0%)
  VPU                 : 0.0000e+00 J  (  0.0%)
Latency breakdown:
  Sc

In [ ]:
print("GLB Capacity Compari

## P=2 set up

In [80]:
qk_spec_base = Spec.from_yaml(
    "../arches/tpu_v4i_with_VPU_QK.yaml",
    "../workloads/C_2/flash_attention_C_2_QK.yaml"
)
qk_results_base = qk_spec_base.map_workload_to_arch()

sm_spec_base = Spec.from_yaml(
    "../arches/tpu_v4i_with_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_SM.yaml"
)
sm_results_base = sm_spec_base.map_workload_to_arch()

av_spec_base = Spec.from_yaml(
    "../arches/tpu_v4i_with_VPU_AV.yaml",
    "../workloads/C_2/flash_attention_C_2_AV.yaml"
)
av_results_base = av_spec_base.map_workload_to_arch()

acc_spec_base = Spec.from_yaml(
    "../arches/tpu_v4i_only_VPU.yaml",
    "../workloads/C_2/flash_attention_C_2_ACC.yaml"
)
acc_results_base = acc_spec_base.map_workload_to_arch()

Getting energy, latency, and leak power for components running QK_1: 100%|████████████████| 1/1 [00:00<00:00, 39.04it/s]
Generating jobs:   0%|                                                                            | 0/1 [00:00<?, ?it/s]
Generating pmapping templates for compute ScalarUnit Einsum QK_1: 0it [00:00, ?it/s]

Generating pmapping templates for compute MXU Einsum QK_1: 0it [00:00, ?it/s]
Generating pmapping templates for compute MXU Einsum QK_1: 32it [00:00, 191.87it/s]
Generating jobs: 100%|████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  4.87it/s]


Einsum QK_1 has 32 pmapping jobs:
	0	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	1	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-h  T-m  T-m_chunk  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	2	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] S-reuse_weight2-e  S-reuse_weight1-m_chunk  [K_1 in MxuBuffer] T-m  [QK_1 in MxuBuffer] T-e  MXU computes QK_1
	3	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m_chunk  [K_1 in GlobalBuffer] T-m  S-reuse_weight2-e  S-reuse_weight1-m_chunk  [QK_1 in MxuBuffer] T-e  [K_1 in MxuBuffer] T-m  MXU computes QK_1
	4	[QK_1 in MainMemory] [Q in MainMemory] [K_1 in MainMemory] T-b  T-e  T-h  T-m  [Q in GlobalBuffer] T-m_chunk  S-reuse_

Compressing pmappings: 100%|█████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 207.20it/s]


Dirty joining with resource usage <= 1.02× optimal
Dirty joining with objectives <= 2× optimal


Dirty pruning pmappings: 100%|██████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1467.57it/s]


...

Filtering out pmappings worse than the following:
	Total<SEP>energy=2.84e-06
Final clean join.


Dirty pruning pmappings: 100%|██████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1690.57it/s]


Dirty joining uses 100.00% of the pmappings
Filtered 1 -> 1 (100.00% kept) pmappings


Final consolidate: 100%|███████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 11428.62it/s]


Dirty joining mapping(s) valid & optimal! Returning...


## 25% GLB Capacity

## 50% GLB Capacity

## 200% GLB Capacity

## Comparisons

In [85]:
# Compare max tensor footprint vs GLB sizes
M_CHUNK, H, E = 4096, 128, 128
bytes_per_val = 1  # 8-bit

# Largest tensor in QK stage: K_1[M_CHUNK, H, E]
k1_bytes = M_CHUNK * H * E * bytes_per_val
print(f"K_1 size: {k1_bytes / 1e6:.1f} MB")

glbs = {
    "25%":  1024*1024*128*2,
    "50%":  1024*1024*128*4,
    "100%": 1024*1024*128*8,
    "200%": 1024*1024*128*16,
}
for name, size in glbs.items():
    print(f"GLB {name}: {size/1e6:.0f} MB  — K_1 fits: {k1_bytes < size}")

K_1 size: 67.1 MB
GLB 25%: 268 MB  — K_1 fits: True
GLB 50%: 537 MB  — K_1 fits: True
GLB 100%: 1074 MB  — K_1 fits: True
GLB 200%: 2147 MB  — K_1 fits: True


## 25%

## 50%

## 200%

## Comparison

In [90]:
# Check whether the workload is bandwidth-bound or compute-bound
# If compute time >> memory transfer time, bandwidth doesn't matter

M_CHUNK, H, E = 4096, 128, 128
bytes_per_val = 1  # 8-bit

# Total bytes transferred for QK stage (read Q, K_1; write QK_1)
# Q[1, 1, H, E], K_1[1, M_CHUNK, H, E], QK_1[1, 1, M_CHUNK, H]
q_bytes   = 1 * 1 * H * E * bytes_per_val
k1_bytes  = 1 * M_CHUNK * H * E * bytes_per_val
qk1_bytes = 1 * 1 * M_CHUNK * H * bytes_per_val
total_bw_bytes = q_bytes + k1_bytes + qk1_bytes
print(f"QK stage total data transferred: {total_bw_bytes / 1e6:.1f} MB")

# MXU compute: Q * K_1 = 1 x H x E * M_CHUNK ops → M_CHUNK * H * E MACs
mxu_ops = M_CHUNK * H * E
mxu_throughput = 128 * 128 * 2 * 1.05e9  # 128x128 MXU @ 1.05 GHz (ops/s)
compute_time_s = mxu_ops / mxu_throughput
print(f"QK compute time: {compute_time_s * 1e6:.3f} us")

bandwidths = {
    "25%  (512 GB/s)":  512e9,
    "50%  (1024 GB/s)": 1024e9,
    "100% (2048 GB/s)": 2048e9,
    "200% (4096 GB/s)": 4096e9,
}
print()
for name, bw in bandwidths.items():
    mem_time_s = total_bw_bytes / bw
    print(f"BW {name}: mem transfer = {mem_time_s * 1e6:.3f} us  — "
          f"{'MEMORY-BOUND' if mem_time_s > compute_time_s else 'COMPUTE-BOUND (BW doesnt matter)'}")

""""
Got to figure out how to make this rely on GLB instead of DRAM
The workload is memory-bound at DRAM, not GLB. Changing GLB bandwidth
(512→4096 GB/s) has no effect because the DRAM bottleneck dominates
everything downstream. To actually see a difference, you'd need to sweep
MainMemory bandwidth instead.

The reason we are compute-bound is because we have high reuse
which allows. If we don't have flash attention, we need ot compute our entire S
If we do have flash atteniton, we can partioin our softmax computation which
gives us more opportunities for reuse. A lot of our intermediate stuff
is put in our buffer allowing us to fuse through our softmax



"""

QK stage total data transferred: 67.6 MB
QK compute time: 1.950 us

BW 25%  (512 GB/s): mem transfer = 132.128 us  — MEMORY-BOUND
BW 50%  (1024 GB/s): mem transfer = 66.064 us  — MEMORY-BOUND
BW 100% (2048 GB/s): mem transfer = 33.032 us  — MEMORY-BOUND
BW 200% (4096 GB/s): mem transfer = 16.516 us  — MEMORY-BOUND


'"\nGot to figure out how to make this rely on GLB instead of DRAM\nThe workload is memory-bound at DRAM, not GLB. Changing GLB bandwidth\n(512→4096 GB/s) has no effect because the DRAM bottleneck dominates\neverything downstream. To actually see a difference, you\'d need to sweep\nMainMemory bandwidth instead.\n\nThe reason we are compute-bound is because we have high reuse\nwhich allows. If we don\'t have flash attention, we need ot compute our entire S\nIf we do have flash atteniton, we can partioin our softmax computation which\ngives us more opportunities for reuse. A lot of our intermediate stuff\nis put in our buffer allowing us to fuse through our softmax\n\n\n\n'

## 25%

## 50%

## 200%

## Comparison

# bandwidth/mxu cycle checks

In [110]:
def print_traffic_breakdown(result, label=""):
    """Print per-component energy and latency breakdown for a mapping result."""
    if label:
        print(f"=== {label} ===")

    # Per-component energy
    energy_breakdown = result.energy(per_component=True)
    total_energy = float(result.energy())
    print("Energy breakdown:")
    for comp, val in sorted(energy_breakdown.items(), key=lambda x: -float(x[1])):
        fval = float(val)
        pct = 100 * fval / total_energy if total_energy > 0 else 0
        print(f"  {comp:<20s}: {fval:.4e} J  ({pct:5.1f}%)")

    # Per-component latency
    latency_breakdown = result.latency(per_component=True)
    total_latency = float(result.latency())
    print("Latency breakdown:")
    for comp, val in sorted(latency_breakdown.items(), key=lambda x: -float(x[1])):
        fval = float(val)
        pct = 100 * fval / total_latency if total_latency > 0 else 0
        print(f"  {comp:<20s}: {fval:.4e} s  ({pct:5.1f}%)")
    print()


def print_pipeline_breakdown(qk_results, sm_results, av_results, acc_results, label=""):
    """Print traffic breakdown for all 4 pipeline stages."""
    if label:
        print(f"{'='*10} {label} {'='*10}")
    print_traffic_breakdown(qk_results, "QK stage")
    print_traffic_breakdown(sm_results, "SM (softmax) stage")
    print_traffic_breakdown(av_results, "AV stage")
    print_traffic_breakdown(acc_results, "ACC (accumulate) stage")


# Run on baseline to see where time/energy actually goes
print_pipeline_breakdown(qk_results_base, sm_results_base, av_results_base, acc_results_base, "Baseline")


========== Baseline ==========
=== QK stage ===
Energy breakdown:
  MainMemory          : 3.8046e-03 J  ( 86.5%)
  MxuBuffer           : 5.8197e-04 J  ( 13.2%)
  GlobalBuffer        : 8.1946e-06 J  (  0.2%)
  MXU                 : 5.6371e-06 J  (  0.1%)
  ScalarBuffer        : 0.0000e+00 J  (  0.0%)
  ScalarUnit          : 0.0000e+00 J  (  0.0%)
Latency breakdown:
  MainMemory          : 1.1018e-04 s  (100.0%)
  MXU                 : 3.9010e-06 s  (  3.5%)
  GlobalBuffer        : 2.5600e-07 s  (  0.2%)
  MxuBuffer           : 0.0000e+00 s  (  0.0%)

=== SM (softmax) stage ===
Energy breakdown:
  MainMemory          : 8.8472e-05 J  ( 65.6%)
  GlobalBuffer        : 2.5669e-05 J  ( 19.0%)
  ScalarBuffer        : 1.8187e-05 J  ( 13.5%)
  ScalarUnit          : 2.5166e-06 J  (  1.9%)
  MxuBuffer           : 0.0000e+00 J  (  0.0%)
  MXU                 : 0.0000e+00 J  (  0.0%)
  VpuBuffer           : 0.0000e+00 J  (  0.0%)
  VPU                 : 0.0000e+00 J  (  0.0%)
Latency breakdown:
  Sc

In [112]:
print_pipeline_breakdown(qk_results_200_band, sm_results_200_band, av_results_200_band, acc_results_200_band, "200 band")


========== 200 band ==========
=== QK stage ===
Energy breakdown:
  MainMemory          : 3.8046e-03 J  ( 86.5%)
  MxuBuffer           : 5.8197e-04 J  ( 13.2%)
  GlobalBuffer        : 8.1946e-06 J  (  0.2%)
  MXU                 : 5.6371e-06 J  (  0.1%)
  ScalarBuffer        : 0.0000e+00 J  (  0.0%)
  ScalarUnit          : 0.0000e+00 J  (  0.0%)
Latency breakdown:
  MainMemory          : 5.5089e-05 s  (100.0%)
  MXU                 : 3.9010e-06 s  (  7.1%)
  GlobalBuffer        : 1.2800e-07 s  (  0.2%)
  MxuBuffer           : 0.0000e+00 s  (  0.0%)

=== SM (softmax) stage ===
Energy breakdown:
  MainMemory          : 8.8472e-05 J  ( 65.6%)
  GlobalBuffer        : 2.5669e-05 J  ( 19.0%)
  ScalarBuffer        : 1.8187e-05 J  ( 13.5%)
  ScalarUnit          : 2.5166e-06 J  (  1.9%)
  MxuBuffer           : 0.0000e+00 J  (  0.0%)
  MXU                 : 0.0000e+00 J  (  0.0%)
  VpuBuffer           : 0.0000e+00 J  (  0.0%)
  VPU                 : 0.0000e+00 J  (  0.0%)
Latency breakdown:
  Sc